In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms

from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Using device:",
    device
)

In [ ]:
def find_project_root():

    current = Path.cwd().resolve()

    while current != current.parent:

        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError(
        "Could not find project root."
    )


PROJECT_ROOT = find_project_root()

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "gtsrb"
)

SPLIT_DIR = (
    PROJECT_ROOT
    / "splits"
)

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [ ]:
from Common.dataset import (
    create_dataloaders
)

from Common.training import (
    train_model
)

from Common.evaluation import (
    get_predictions,
    calculate_metrics,
    plot_training_curves,
    plot_confusion_matrix,
    print_classification_report,
    total_parameters,
    trainable_parameters,
    save_model_and_get_size,
    measure_inference_time
)

In [ ]:
train_df = pd.read_csv(
    SPLIT_DIR
    / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR
    / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR
    / "test_split.csv"
)

In [ ]:
imagenet_train_transform = transforms.Compose([
    transforms.Resize(
        (64, 64)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.RandomRotation(
        degrees=8
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


imagenet_eval_transform = transforms.Compose([
    transforms.Resize(
        (64, 64)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
train_loader, val_loader, test_loader = (
    create_dataloaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        data_root=DATA_ROOT,
        batch_size=32,
        num_workers=0,
        train_transform_override=imagenet_train_transform,
        eval_transform_override=imagenet_eval_transform,
        seed=SEED
    )
)

In [ ]:
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

In [ ]:
torch.hub.set_dir(str(PROJECT_ROOT / "checkpoints" / "torch_hub"))

weights = (
    EfficientNet_B0_Weights.DEFAULT
)

efficient_model = efficientnet_b0(
    weights=weights
)

In [ ]:
for parameter in efficient_model.parameters():
    parameter.requires_grad = False

In [ ]:
input_features = (
    efficient_model
    .classifier[1]
    .in_features
)

efficient_model.classifier[1] = (
    nn.Linear(
        input_features,
        43
    )
)

efficient_model = (
    efficient_model.to(
        device
    )
)

Classifier head training

In [ ]:
criterion = nn.CrossEntropyLoss()

efficient_optimizer = optim.Adam(
    filter(
        lambda parameter:
        parameter.requires_grad,
        efficient_model.parameters()
    ),
    lr=1e-3
)

efficient_model, efficient_head_history = (
    train_model(
        model=efficient_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=efficient_optimizer,
        criterion=criterion,
        epochs=10,
        device=device
    )
)

fine tuning

In [ ]:
for parameter in (
    efficient_model
    .features[-2:]
    .parameters()
):
    parameter.requires_grad = True

In [ ]:
efficient_optimizer_ft = optim.Adam(
    filter(
        lambda parameter:
        parameter.requires_grad,
        efficient_model.parameters()
    ),
    lr=1e-5
)

Train

In [ ]:
efficient_model, efficient_ft_history = (
    train_model(
        model=efficient_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=efficient_optimizer_ft,
        criterion=criterion,
        epochs=20,
        device=device
    )
)

In [ ]:
efficient_history = {}

for key in efficient_head_history.keys():

    efficient_history[key] = (
        efficient_head_history[key]
        + efficient_ft_history[key]
    )

In [ ]:
FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

plot_training_curves(
    efficient_history,
    "EfficientNetB0",
    FIGURES_DIR
)

Evaluate

In [ ]:
y_true_eff, y_pred_eff = (
    get_predictions(
        efficient_model,
        test_loader,
        device
    )
)

efficient_metrics = (
    calculate_metrics(
        y_true_eff,
        y_pred_eff
    )
)

print(
    efficient_metrics
)

In [ ]:
plot_confusion_matrix(
    y_true_eff,
    y_pred_eff,
    "EfficientNetB0",
    FIGURES_DIR
)

In [ ]:
print_classification_report(
    y_true_eff,
    y_pred_eff
)

In [ ]:
efficient_total_params = (
    total_parameters(
        efficient_model
    )
)

efficient_model_path = (
    PROJECT_ROOT
    / "checkpoints"
    / "efficientnetb0_final.pth"
)

efficient_size_kb = (
    save_model_and_get_size(
        efficient_model,
        efficient_model_path
    )
)

efficient_size_mb = (
    efficient_size_kb
    / 1024
)

efficient_inference = (
    measure_inference_time(
        efficient_model,
        test_loader,
        device
    )
)

In [ ]:
(PROJECT_ROOT / "results" / "metrics").mkdir(parents=True, exist_ok=True)

efficient_results = {
    "Model": "EfficientNetB0",
    "Parameters": efficient_total_params,
    "Model Size (MB)": efficient_size_mb,
    "Inference Time (ms/image)": efficient_inference,
    "Test Accuracy": efficient_metrics["Test Accuracy"],
    "Macro Precision": efficient_metrics["Macro Precision"],
    "Macro Recall": efficient_metrics["Macro Recall"]
}

pd.DataFrame(
    [efficient_results]
).to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "efficientnetb0_metrics.csv",
    index=False
)